In [1]:
import warnings
import os
warnings.simplefilter(action='ignore')
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
#parameters

### USER EDIT start
run_dir = '/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params'
esm_file= os.path.join(run_dir, 'cm3-demo-datastore/cm3-demo-datastore.json')
plotfolder='/g/data/tm70/ek4684/access-om3-paper-1/notebooks/mkfigs_output4/'
dpi=300
### USER EDIT stop

import matplotlib as mpl
import os
%matplotlib inline
mpl.rcParams['figure.dpi']= dpi

os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ",esm_file)
print("Plot folder path: ",plotfolder)

ESM datastore path:  /g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/cm3-demo-datastore/cm3-demo-datastore.json
Plot folder path:  /g/data/tm70/ek4684/access-om3-paper-1/notebooks/mkfigs_output4/


In [3]:
import xarray as xr
import cf_xarray as cfxr
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import numpy as np

In [4]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

/proxy/8787/status


In [5]:
#datastore_path = "/g/data/ol01/access-om3-output/access-om3-025/MC_25km_jra_ryf-1.0-beta/experiment_datastore.json"
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

In [6]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [7]:
# datastore_filtered = datastore.search(realm="ocean", frequency="1mon")

# for col in COLUMNS_WITH_ITERABLES:
#     datastore_filtered.df[col] = datastore_filtered.df[col].astype("object").map(lambda x: list(map(str, x)))

# df = available_variables(datastore_filtered)

## Load ocean and atmosphere areas

In [8]:
wet = datastore.search(variable="wet").to_dask().compute()
areacello = datastore.search(variable="areacello").to_dask().compute()

areacello = (areacello.areacello * (wet.wet == 1.0))

In [ ]:
# areacella = 

In [9]:
EARTH_RADIUS = 6371229.0
nlon, nlat = 192, 144
dx = 360 / nlon
dy = 180 / nlat

element_lat = (np.arange(nlat) + 0.5) * dy  - 90
element_lat = element_lat[:, None] * np.ones(nlon)[None, :]

pi_over_180 = np.pi / 180
element_areas = dx * pi_over_180 * (
  np.sin((element_lat + 0.5 * dy) * pi_over_180) - np.sin((element_lat - 0.5 * dy) * pi_over_180)
)

areacella = element_areas * EARTH_RADIUS**2
areacella = xr.DataArray(areacella, coords=dict(lat=toa.coords['lat'], lon=toa.coords['lon']), dims=('lat', 'lon'))

total_ocn_fraction = areacello.sum().data / areacella.sum().data

NameError: name 'toa' is not defined

## Ocean time series

### Ocean heatflux

In [ ]:
ocean_heatflux = datastore.search(variable="net_heat_coupler", frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
).net_heat_coupler

In [ ]:
# area weighted average of ocean heatflux
# average over entire earth for comparison with TOA
avg_ocean_heatflux = ocean_heatflux.weighted(areacello.fillna(0)).mean(dim=('yh', 'xh')).compute() * total_ocn_fraction

In [ ]:
N = 12
rolling_avg = np.convolve(avg_ocean_heatflux, np.ones(N) / N, mode='valid')

plt.title('Global average heat flux into ocean (12 month rolling average)')
plt.plot(np.arange(len(rolling_avg)) / 12, rolling_avg)
plt.ylabel('Heat flux (W/m$^2$)')
plt.xlabel('Years')

### SST

In [ ]:
sst = datastore.search(variable="tos", frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
).tos

In [ ]:
# area weighted average of SST
avg_sst = sst.weighted(areacello.fillna(0)).mean(dim=('yh', 'xh')).compute()

In [ ]:
N = 12
rolling_avg = np.convolve(avg_sst, np.ones(N) / N, mode='valid')

plt.title('Average SST (12 month rolling average)')
plt.plot(np.arange(len(rolling_avg)) / 12, rolling_avg)
plt.ylabel('Heat flux (W/m$^2$)')
plt.xlabel('Years')

## Ice time series

In [17]:
ice_filepaths = []
for year in range(1981, 1991):
    for month in range (1, 13):
        fp = os.path.join(run_dir, 'archive', str(year), 'ice', f'access-cm3.cice.1mon.mean.{year}-{month:02d}.nc')
        ice_filepaths.append(fp)

    break

In [ ]:
# %%time
# ice_ds = xr.open_mfdataset(
#     ice_filepaths, decode_times=False
# )
1

In [18]:
%%time
ice_ds_list = []
for fp in ice_filepaths:
    ice_ds_list.append(xr.open_dataset(fp))



CPU times: user 14.4 s, sys: 6.55 s, total: 20.9 s
Wall time: 31.4 s


In [ ]:
ice_ds = xr.concat(ice_ds_list, dim='time')

In [176]:
ice_ds

'/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-11-08-2025-25km-beta-om3-new-um-params/archive/ice/*/access-cm3.cice.1mon.mean.nc'

In [121]:
ice_fraction = datastore.search(variable="aice_m", frequency="1mon")

## Atmosphere time series

In [18]:
sw_out_ds = datastore.search(variable="fld_s01i208", frequency="1mon")
lw_out_ds = datastore.search(variable="fld_s02i205", frequency="1mon")
sw_in_ds = datastore.search(variable="fld_s01i207", frequency="1mon")

In [19]:
# sw_out_ds.df.variable_standard_name[0]

In [20]:
xarray_combine_by_coords_kwargs={
    'compat' : 'override',
    'data_vars': 'minimal',
    'coords': 'minimal'
}

sw_out = sw_out_ds.to_dask(
    xarray_open_kwargs={
        "chunks": "auto",
    },
    xarray_combine_by_coords_kwargs=xarray_combine_by_coords_kwargs
).fld_s01i208

lw_out = lw_out_ds.to_dask(
        xarray_open_kwargs= {
        "chunks": "auto",
    },
    xarray_combine_by_coords_kwargs=xarray_combine_by_coords_kwargs
).fld_s02i205

sw_in = sw_in_ds.to_dask(
        xarray_open_kwargs= {
        "chunks": "auto",
    },
    xarray_combine_by_coords_kwargs=xarray_combine_by_coords_kwargs
).fld_s01i207

2025-10-20 16:33:05,175 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 3.29 GiB -- Worker memory limit: 4.50 GiB
2025-10-20 16:33:06,704 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 3.40 GiB -- Worker memory limit: 4.50 GiB
2025-10-20 16:43:00,741 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- U

In [46]:
toa = sw_in - sw_out - lw_out

In [ ]:
avg_toa = toa.weighted(areacella).mean(dim=('lat', 'lon')).compute()

In [41]:
N = 12
rolling_avg = np.convolve(avg_ocean_heatflux, np.ones(N) / N, mode='valid', label='Ocean')
plt.plot(np.arange(len(rolling_avg)) / 12, rolling_avg)

rolling_avg = np.convolve(avg_toa, np.ones(N) / N, mode='valid', label='TOA')
plt.plot(np.arange(len(rolling_avg)) / 12, rolling_avg)

plt.title('Global average heat flux')

plt.ylabel('Heat flux (W/m$^2$)')
plt.xlabel('Years')
plt.legend()

<xarray.Dataset> Size: 0B
Dimensions:  ()
Data variables:
    *empty*